# Train LightGBM Model Using SPARCS 2024

This notebook trains a LightGBM model to predict prolonged length of stay using the 2024 SPARCS inpatient discharge dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# lightgbm - for train LightGBM model
# shap - for XAI use
# joblib - for save pipeline

!pip install -q lightgbm shap joblib

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from lightgbm import LGBMClassifier

In [ ]:
BASE_DIR = "/content/drive/MyDrive/FYP/SPARCS"

PROCESSED_PATH = f"{BASE_DIR}/processed/sparcs_2024_processed.csv"
SPLIT_DIR = f"{BASE_DIR}/splits"
MODEL_DIR = f"{BASE_DIR}/models"
RESULT_DIR = f"{BASE_DIR}/results"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

In [ ]:
# Load Processed Dataset and Split Files

df = pd.read_csv(PROCESSED_PATH, low_memory=False)

train_idx = np.load(f"{SPLIT_DIR}/train_idx.npy")
val_idx = np.load(f"{SPLIT_DIR}/val_idx.npy")
test_idx = np.load(f"{SPLIT_DIR}/test_idx.npy")

feature_columns = joblib.load(f"{SPLIT_DIR}/feature_columns.joblib")
feature_schema = joblib.load(f"{SPLIT_DIR}/feature_schema.joblib")

print("Dataset shape:", df.shape)
print("Number of selected features:", len(feature_columns))
print(feature_columns)

Dataset shape: (2196737, 35)
Number of selected features: 13
['Age Group', 'Gender', 'Race', 'Ethnicity', 'Type of Admission', 'CCSR Diagnosis Description', 'CCSR Procedure Description', 'APR DRG Description', 'APR MDC Description', 'APR Severity of Illness Description', 'APR Risk of Mortality', 'APR Medical Surgical Description', 'Emergency Department Indicator']


In [ ]:
# Prepare X and y

target_col = "prolonged_los"

X = df[feature_columns].copy()
y = df[target_col].copy()

X_train = X.iloc[train_idx]
y_train = y.iloc[train_idx]

X_val = X.iloc[val_idx]
y_val = y.iloc[val_idx]

X_test = X.iloc[test_idx]
y_test = y.iloc[test_idx]

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (1405911, 13) (1405911,)
Validation: (351478, 13) (351478,)
Test: (439348, 13) (439348,)


In [ ]:
# Check Class Distribution

class_distribution = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "positive_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
    "positive_count": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
    "total_count": [len(y_train), len(y_val), len(y_test)]
})

class_distribution

,split,positive_rate,positive_count,total_count
0,train,0.241135,339015,1405911
1,validation,0.241136,84754,351478
2,test,0.241137,105943,439348


In [ ]:
# Identify Categorical and Numeric Columns

cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:", cat_cols)
print("Numeric columns:", num_cols)

Categorical columns: ['Age Group', 'Gender', 'Race', 'Ethnicity', 'Type of Admission', 'CCSR Diagnosis Description', 'CCSR Procedure Description', 'APR DRG Description', 'APR MDC Description', 'APR Severity of Illness Description', 'APR Risk of Mortality', 'APR Medical Surgical Description', 'Emergency Department Indicator']
Numeric columns: []


In [ ]:
# Build Preprocessing Pipeline
# Although LighGBM can handle categorical features in some setups
# To be fair to XGBoost, we'll use the same One-Hot preprocessing for now.

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

In [ ]:
# Handle Class Imbalance

positive_count = y_train.sum()
negative_count = len(y_train) - positive_count

scale_pos_weight = negative_count / positive_count

print("Positive count:", positive_count)
print("Negative count:", negative_count)
print("scale_pos_weight:", scale_pos_weight)

Positive count: 339015
Negative count: 1066896
scale_pos_weight: 3.1470465908588117


In [ ]:
# Train LightGBM Model
# 'num_leaves' controls the tree complexity
# 'scale_pos_weight' is used to handle prolonged LOS imbalance.


lgbm_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)

lgbm_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", lgbm_model)
])

lgbm_pipeline.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 339015, number of negative: 1066896
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.155534 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2300
[LightGBM] [Info] Number of data points in the train set: 1405911, number of used features: 1150
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.241135 -> initscore=-1.146464
[LightGBM] [Info] Start training from score -1.146464


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  []),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Age Group', 'Gender',
                                                   'Race', 'Ethnicity',
                                                   'Type of Admission',
                                                   'CCSR Diagnosis Description',
                                                   'CCSR Procedure Description',
                                                   'APR DRG Description',
                                                   'APR MDC Description',
                                                   'APR Severity of Illness '
                                                   'Description',
                                                   'APR Risk of Mortality',
                                                   'APR Medical Surgical '
                                                   'Description',
                                                   'Emergency Department '
                                                   'Indicator'])])),
                ('model',
                 LGBMClassifier(colsample_bytree=0.8, learning_rate=0.03,
                                n_estimators=500, n_jobs=-1, random_state=42,
                                scale_pos_weight=np.float64(3.1470465908588117),
                                subsample=0.8))])

In [ ]:
# Evaluation Function
# ROC-AUC: ranking ability
# PR-AUC: performance on positive class under imbalance
# Recall: ability to detect prolonged LOS cases
# F1: balance of precision and recall


def evaluate_binary_classifier(model, X_data, y_true, split_name, threshold=0.5):
    y_prob = model.predict_proba(X_data)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "model": "LightGBM",
        "dataset": "SPARCS 2024",
        "split": split_name,
        "threshold": threshold,
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0)
    }

    return metrics, y_prob, y_pred

In [ ]:
# Validate Model

val_metrics, y_prob_val, y_pred_val = evaluate_binary_classifier(
    lgbm_pipeline,
    X_val,
    y_val,
    "validation",
    threshold=0.5
)

pd.DataFrame([val_metrics])

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,model,dataset,split,threshold,roc_auc,pr_auc,accuracy,precision,recall,f1
0,LightGBM,SPARCS 2024,validation,0.5,0.879007,0.712681,0.783901,0.534172,0.811525,0.644267


In [ ]:
print(confusion_matrix(y_val, y_pred_val))
print(classification_report(y_val, y_pred_val))

[[206744  59980]
 [ 15974  68780]]
              precision    recall  f1-score   support

           0       0.93      0.78      0.84    266724
           1       0.53      0.81      0.64     84754

    accuracy                           0.78    351478
   macro avg       0.73      0.79      0.74    351478
weighted avg       0.83      0.78      0.80    351478



In [ ]:
# Threshold Tuning

threshold_results = []

for threshold in np.arange(0.20, 0.81, 0.05):
    metrics, _, _ = evaluate_binary_classifier(
        lgbm_pipeline,
        X_val,
        y_val,
        "validation",
        threshold=threshold
    )
    threshold_results.append(metrics)

threshold_df = pd.DataFrame(threshold_results)

threshold_df.sort_values(by="f1", ascending=False).head(10)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

,model,dataset,split,threshold,roc_auc,pr_auc,accuracy,precision,recall,f1
8,LightGBM,SPARCS 2024,validation,0.60,0.879007,0.712681,0.817730,0.601581,0.722857,0.656666
9,LightGBM,SPARCS 2024,validation,0.65,0.879007,0.712681,0.828075,0.635069,0.674753,0.654310
7,LightGBM,SPARCS 2024,validation,0.55,0.879007,0.712681,0.803717,0.568978,0.767161,0.653372
6,LightGBM,SPARCS 2024,validation,0.50,0.879007,0.712681,0.783901,0.534172,0.811525,0.644267
10,LightGBM,SPARCS 2024,validation,0.70,0.879007,0.712681,0.835276,0.673573,0.614850,0.642873
5,LightGBM,SPARCS 2024,validation,0.45,0.879007,0.712681,0.763308,0.505514,0.844833,0.632541
4,LightGBM,SPARCS 2024,validation,0.40,0.879007,0.712681,0.741406,0.480083,0.872561,0.619382
11,LightGBM,SPARCS 2024,validation,0.75,0.879007,0.712681,0.837617,0.717153,0.539290,0.615632
3,LightGBM,SPARCS 2024,validation,0.35,0.879007,0.712681,0.712733,0.451959,0.899887,0.601714
2,LightGBM,SPARCS 2024,validation,0.30,0.879007,0.712681,0.679274,0.424300,0.925006,0.581752


# Select Deployment Threshold

The threshold with the best validation F1-score is identified for reference.  
The final deployment threshold is selected by prioritizing recall for prolonged LOS screening, because missed prolonged LOS cases are more clinically important than some false positives.

In [ ]:
best_f1_threshold_row = threshold_df.sort_values(by="f1", ascending=False).iloc[0]
best_f1_threshold = float(best_f1_threshold_row["threshold"])

best_f1_threshold_row

,8
model,LightGBM
dataset,SPARCS 2024
split,validation
threshold,0.6
roc_auc,0.879007
pr_auc,0.712681
accuracy,0.81773
precision,0.601581
recall,0.722857
f1,0.656666


In [ ]:
candidate_thresholds = threshold_df[threshold_df["recall"] >= 0.70]

if len(candidate_thresholds) > 0:
    deployment_threshold_row = candidate_thresholds.sort_values(
        by=["f1", "precision"],
        ascending=False
    ).iloc[0]
else:
    deployment_threshold_row = best_f1_threshold_row

deployment_threshold = float(deployment_threshold_row["threshold"])

deployment_threshold_row

,8
model,LightGBM
dataset,SPARCS 2024
split,validation
threshold,0.6
roc_auc,0.879007
pr_auc,0.712681
accuracy,0.81773
precision,0.601581
recall,0.722857
f1,0.656666


In [ ]:
# Final Validation and Test Evaluation

val_metrics_deploy, y_prob_val, y_pred_val = evaluate_binary_classifier(
    lgbm_pipeline,
    X_val,
    y_val,
    "validation",
    threshold=deployment_threshold
)

test_metrics_deploy, y_prob_test, y_pred_test = evaluate_binary_classifier(
    lgbm_pipeline,
    X_test,
    y_test,
    "test",
    threshold=deployment_threshold
)

metrics_df = pd.DataFrame([val_metrics_deploy, test_metrics_deploy])
metrics_df

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,model,dataset,split,threshold,roc_auc,pr_auc,accuracy,precision,recall,f1
0,LightGBM,SPARCS 2024,validation,0.6,0.879007,0.712681,0.81773,0.601581,0.722857,0.656666
1,LightGBM,SPARCS 2024,test,0.6,0.878120,0.711166,0.81748,0.601391,0.720916,0.655751


In [ ]:
# Summary Check

positive_baseline = y_train.mean()

summary_check = pd.DataFrame([
    {
        "metric": "Positive baseline",
        "value": positive_baseline,
        "requirement": "PR-AUC should be higher than this"
    },
    {
        "metric": "Validation ROC-AUC",
        "value": val_metrics_deploy["roc_auc"],
        "requirement": ">= 0.70"
    },
    {
        "metric": "Validation PR-AUC",
        "value": val_metrics_deploy["pr_auc"],
        "requirement": "> positive baseline"
    },
    {
        "metric": "Validation Recall",
        "value": val_metrics_deploy["recall"],
        "requirement": ">= 0.65 preferred"
    },
    {
        "metric": "Test ROC-AUC",
        "value": test_metrics_deploy["roc_auc"],
        "requirement": ">= 0.70"
    },
    {
        "metric": "Test PR-AUC",
        "value": test_metrics_deploy["pr_auc"],
        "requirement": "> positive baseline"
    },
    {
        "metric": "Test Recall",
        "value": test_metrics_deploy["recall"],
        "requirement": ">= 0.65 preferred"
    }
])

summary_check

,metric,value,requirement
0,Positive baseline,0.241135,PR-AUC should be higher than this
1,Validation ROC-AUC,0.879007,>= 0.70
2,Validation PR-AUC,0.712681,> positive baseline
3,Validation Recall,0.722857,>= 0.65 preferred
4,Test ROC-AUC,0.878120,>= 0.70
5,Test PR-AUC,0.711166,> positive baseline
6,Test Recall,0.720916,>= 0.65 preferred


In [ ]:
# Risk Level Mapping

def risk_level(probability):
    if probability < 0.33:
        return "Low"
    elif probability < 0.66:
        return "Medium"
    else:
        return "High"

In [ ]:
sample_output = pd.DataFrame({
    "predicted_probability": y_prob_test[:10],
    "prolonged_los_prediction": (y_prob_test[:10] >= deployment_threshold).astype(int),
    "risk_level": [risk_level(p) for p in y_prob_test[:10]]
})

sample_output

,predicted_probability,prolonged_los_prediction,risk_level
0,0.476619,0,Medium
1,0.085560,0,Low
2,0.306265,0,Low
3,0.327093,0,Low
4,0.910397,1,High
5,0.008351,0,Low
6,0.965093,1,High
7,0.882104,1,High
8,0.150828,0,Low
9,0.151879,0,Low


In [ ]:
# Save Model, Metrics, Threshold Result, Metadata

lgbm_model_path = f"{MODEL_DIR}/lightgbm_sparcs_los_pipeline.joblib"
joblib.dump(lgbm_pipeline, lgbm_model_path)

metrics_df.to_csv(f"{RESULT_DIR}/lightgbm_sparcs_metrics.csv", index=False)
threshold_df.to_csv(f"{RESULT_DIR}/lightgbm_sparcs_threshold_tuning.csv", index=False)

metadata = {
    "model_name": "LightGBM",
    "dataset": "SPARCS 2024",
    "target": "prolonged_los",
    "target_definition": "Length of Stay >= 7 days",
    "selected_threshold": deployment_threshold,
    "threshold_selection_reason": "Selected on validation set by prioritizing recall >= 0.70 and then F1-score.",
    "best_f1_threshold": best_f1_threshold,
    "risk_level_thresholds": {
        "low": "<0.33",
        "medium": "0.33-0.66",
        "high": ">=0.66"
    },
    "feature_columns": feature_columns,
    "feature_schema": feature_schema
}

joblib.dump(metadata, f"{MODEL_DIR}/lightgbm_sparcs_metadata.joblib")

print("Saved LightGBM pipeline:", lgbm_model_path)
print("Saved metrics and metadata.")

Saved LightGBM pipeline: /content/drive/MyDrive/FYP/SPARCS/models/lightgbm_sparcs_los_pipeline.joblib
Saved metrics and metadata.


## Summary

A LightGBM model was trained using the selected SPARCS 2024 deployment-friendly features.  
The same train, validation, and test split used in the XGBoost experiment was reused to support fair model comparison.  
The model was evaluated using ROC-AUC, PR-AUC, accuracy, precision, recall, and F1-score.  
Threshold tuning was performed on the validation set, and the final threshold was selected by prioritizing recall for prolonged LOS screening.  
The trained pipeline, metrics, threshold tuning results, and metadata were saved for later model comparison and deployment.